
The entire set of scripts (00–03, four scripts in total) processes three main inputs: the original survey data, the reweighted survey trips, and a TAZ-to-district lookup table. The primary goal is to create additional columns that support summarizing and analyzing the survey data. These include trip purpose in Production-Attraction (PA) format, mode of access in PA format, and rider information. The output is a more detailed and comprehensive dataset suitable for further analysis. See `output/Final_2023_OnBoardSurvey_Dataset.csv`.

In addition, the scripts generate high-level summary tables that support the final report. Look for sections labeled `Table xx in Report` to see how each output corresponds to the Mid-Coast After Study Report.


### Structure


- **High-Level Summary**
  - Project Trips Summary  
    - *Table 3 in Report*

- **Station Boardings and Alightings**
  - *Tables 7, 8, and 21 in Report*

- **District-to-District Summary**
  - *Table 6 and Appendix A in Report*

- **Station Portfolio Analysis** *(Section 2.1.2.1 in Report)*
  - Generates final full datasets
  - Reference: `4_2_Rider_Portfolio_by_Station_Archive.xlsx` for the Report


# High Level Summary

**Purpose:** Summarize trips using data from the 2024 SANDAG survey and reweighted records, categorized by trip purpose, auto ownership, and mode of access.

**I/O Files:**

Input Files:

1. `od_20240422_sandagca_weighted_combined_draftfinal.xlsx` - Raw survey file.
2. `2023OBS_ReWeighted_2024-05-24.xlsx` - This includes the reweighted trips for each record.
3. `2023OBS_wTAZ_Districts.csv` - This includes TAZ and District lookup for each record.

Intermediate/Helper Files:

- `Interim_2023_OnBoardSurvey_Dataset_x.csv` : This file includes all survey records, not only the raw survey fields but also the processed columns. It is an intermediate file (an output of this script) that is used as input in the next script for the data model.

Output Files:

- `Table_x.csv`: Summary table used in the report. The table index should match the one in the *Mid-Coast After Survey Report*.

In [1]:
import warnings
warnings.simplefilter("always")

### Read Inputs

In [2]:
## Data Preparation and Summary
import pandas as pd
import numpy as np

# Set directory
data_dir = "../input/"
interim_dir = '../output/interim/'
output_dir = '../output/'

df_full = pd.read_excel(data_dir + "od_20240422_sandagca_weighted_combined_draftfinal.xlsx", sheet_name="OD_RESULTS", engine='openpyxl')

In [3]:
df_weight = pd.read_excel(data_dir + "2023OBS_ReWeighted_2024-05-24.xlsx",
                          sheet_name="Sheet1", engine='openpyxl')

In [4]:
df = pd.merge(df_full, df_weight[['ID', 'REWEIGHTED_LINKED', 'REWEIGHTED_UNLINKED']],
              on='ID', how='left')

In [5]:
df[['AGENCY', 'AGENCY_CODE','ROUTE_NUMBER', 'DIRECTION']] = df['ROUTE_DIRECTION[Code]'].str.split('_', expand=True)

### Define Trip Purpose

In [6]:
# Creating trip purpose: HBW, HBO, and NHB
df['TRIP_PURPOSE'] = np.where(((df['ORIGIN_PLACE_TYPE'] == 'Your HOME') & (df['DESTIN_PLACE_TYPE'] == 'Your usual WORKPLACE')) |
                              ((df['ORIGIN_PLACE_TYPE'] == 'Your usual WORKPLACE') & (df['DESTIN_PLACE_TYPE'] == 'Your HOME')), 'HBW',
                              
                     np.where(((df['ORIGIN_PLACE_TYPE'] == 'Your HOME') & (df['DESTIN_PLACE_TYPE'] == 'School (K-12) (students only)')) |
                              ((df['ORIGIN_PLACE_TYPE'] == 'School (K-12) (students only)') & (df['DESTIN_PLACE_TYPE'] == 'Your HOME')), 'HBSC',                              
                              
                     np.where(((df['ORIGIN_PLACE_TYPE'] == 'Your HOME') & (df['DESTIN_PLACE_TYPE'] == 'College / University (students only)')) |
                              ((df['ORIGIN_PLACE_TYPE'] == 'College / University (students only)') & (df['DESTIN_PLACE_TYPE'] == 'Your HOME')), 'HBU',                               

                     np.where(((df['ORIGIN_PLACE_TYPE'] == 'Your HOME') & (df['DESTIN_PLACE_TYPE'] == 'Medical Services / Hospital (non-work)')) |
                              ((df['ORIGIN_PLACE_TYPE'] == 'Medical Services / Hospital (non-work)') & (df['DESTIN_PLACE_TYPE'] == 'Your HOME')), 'HBM', 
                                       
                     np.where(
                             (
                                (df['ORIGIN_PLACE_TYPE'] == 'Your HOME') & 
                                (
                                    (df['DESTIN_PLACE_TYPE'] != 'Your usual WORKPLACE') &
                                    (df['DESTIN_PLACE_TYPE'] != 'School (K-12) (students only)') &
                                    (df['DESTIN_PLACE_TYPE'] != 'College / University (students only)') &
                                    (df['DESTIN_PLACE_TYPE'] != 'Medical Services / Hospital (non-work)')
                                )
                             ) |
                             (
                                (
                                    (df['ORIGIN_PLACE_TYPE'] != 'Your usual WORKPLACE') &
                                    (df['ORIGIN_PLACE_TYPE'] != 'School (K-12) (students only)') &
                                    (df['ORIGIN_PLACE_TYPE'] != 'College / University (students only)') &
                                    (df['ORIGIN_PLACE_TYPE'] != 'Medical Services / Hospital (non-work)')
                                ) &
                                (df['DESTIN_PLACE_TYPE'] == 'Your HOME')
                             ),
                         'HBO',
                         'NHB')))))

# print(df['TRIP_PURPOSE'].value_counts(dropna=False))

In [7]:
df['TRIP_PURPOSE_AGG'] = np.where(df['TRIP_PURPOSE'].isin(['HBU', 'HBSC', 'HBM']), 'HBO', df['TRIP_PURPOSE'])

In [8]:
# Creating production and attraction flag
# If the trip starts from Production-to-Attraction (PA), then flag=1. If it is Attraction-to-Production (AP), then flag=0.
# Code: If the trip is home-based and the destination is home, the trip is AP format, otherwise it is PA.
df['PA_FLAG'] = np.where(df['TRIP_PURPOSE'].isin(['HBW', 'HBO', 'HBU', 'HBSC', 'HBM']) &
                         df['DESTIN_PLACE_TYPE'].isin(['Your HOME']), 0, 1)

# print(df['PA_FLAG'].value_counts(dropna=False))

### Define Mode of Access in Production-Attraction format

In [9]:
# Creating access mode: Walk, XFER, KNR, and PNR
df['ACCESS_MODE_OD'] = np.where(df['ORIGIN_TRANSPORT'] == 'Walk', 'Walk',
                    
                       np.where(df['ORIGIN_TRANSPORT'].isin(['Was dropped off by someone',
                                                             'Uber, Lyft, etc. (pool or shared)',
                                                             'Taxi',
                                                             'Other shuttle',
                                                             'Uber, Lyft, etc. (private)',
                                                             'Electric vehicle shuttle']), 'KNR',
                        
                       np.where(df['ORIGIN_TRANSPORT'].isin(['Drove alone and parked',
                                                             'Drove or rode with others and parked']), 'PNR', 'Walk')))

df['ACCESS_MODE_wXFER_OD'] = np.where(df['PREV_TRANSFERS[Code]'] > 0, 'XFER', df['ACCESS_MODE_OD'])

In [10]:
df['EGRESS_MODE_OD'] = np.where(df['DESTIN_TRANSPORT'] == 'Walk', 'Walk',
                    
                       np.where(df['DESTIN_TRANSPORT'].isin(['Be picked up by someone',
                                                             'Uber, Lyft, etc. (pool or shared)',
                                                             'Taxi',
                                                             'Other shuttle',
                                                             'Uber, Lyft, etc. (private)',
                                                             'Electric vehicle shuttle']), 'KNR',
                             
                       np.where(df['DESTIN_TRANSPORT'].isin(['Get in a parked vehicle & drive alone',
                                                             'Get in a parked vehicle & drive/ride w/others']), 'PNR', 'Walk')))

df['EGRESS_MODE_wXFER_OD'] = np.where(df['NEXT_TRANSFERS[Code]'] > 0, 'XFER',  df['EGRESS_MODE_OD'])                          

In [11]:
# Adjusting access mode for AP trips
df['ACCESS_MODE_PA'] = np.where(df['PA_FLAG'] == 0, df['EGRESS_MODE_OD'], df['ACCESS_MODE_OD'])
df['ACCESS_MODE_wXFER_PA'] = np.where(df['PA_FLAG'] == 0, df['EGRESS_MODE_wXFER_OD'], df['ACCESS_MODE_wXFER_OD'])

# Adjusting egress mode for AP trips
df['EGRESS_MODE_PA'] = np.where(df['PA_FLAG'] == 0, df['ACCESS_MODE_OD'], df['EGRESS_MODE_OD'])
df['EGRESS_MODE_wXFER_PA'] = np.where(df['PA_FLAG'] == 0, df['ACCESS_MODE_wXFER_OD'], df['EGRESS_MODE_wXFER_OD'])

### Define Auto Ownership

In [12]:
# Creating auto ownership: 0-car, 1-car, 2-car and plus
df['AUTO_OWNERSHIP'] = np.where(df['COUNT_VH_HH'] == 'None (0)', 'None (0)',
                       np.where(df['COUNT_VH_HH'] == 'One (1)', 'One (1)', 'Two or more (2+)'))
                    
# print(df['AUTO_OWNERSHIP'].value_counts(dropna=False))

In [13]:
# Creating a Label identifying the trips-on-project
station_names = [
    'UTC Station',
    'Executive Drive Station',
    'UCSD Health La Jolla Station',
    'UCSD Central Campus Station',
    'VA Medical Center Station',
    'Nobel Drive Station',
    'Balboa Avenue Station',
    'Clairemont Drive Station',
    'Tecolote Road Station'
]

df['TRIPS_ON_PROJECT'] = np.where((df['STOP_ON [ADDR]'].isin(station_names)) | (df['STOP_OFF [ADDR]'].isin(station_names)),'Yes','No')
# print(df['TRIPS_ON_PROJECT'].value_counts(dropna=False))

## Project Trips Summary

In [14]:
top_summary = df.groupby('TRIPS_ON_PROJECT').agg({'REWEIGHTED_LINKED':'sum'}).round(1)
top_summary

,REWEIGHTED_LINKED
TRIPS_ON_PROJECT,
No,203364.4
Yes,22630.5


by trip purpose

In [15]:
project_trips = df[df['TRIPS_ON_PROJECT']=='Yes']
pj_trips_bypurp = project_trips.groupby('TRIP_PURPOSE_AGG').agg({'REWEIGHTED_LINKED': 'sum'}).round(1)
pj_trips_bypurp

,REWEIGHTED_LINKED
TRIP_PURPOSE_AGG,
HBO,11027.4
HBW,7582.5
NHB,4020.6


by auto ownership

In [16]:
pj_trips_byauto = project_trips.groupby('AUTO_OWNERSHIP').agg({'REWEIGHTED_LINKED': 'sum'}).round(1)
pj_trips_byauto

,REWEIGHTED_LINKED
AUTO_OWNERSHIP,
None (0),7708.0
One (1),6743.4
Two or more (2+),8179.2


by mode of access

In [17]:
pj_trips_byaccess = df[df['TRIPS_ON_PROJECT']=='Yes'].groupby('ACCESS_MODE_PA').agg({'REWEIGHTED_LINKED': 'sum'}).round(1)
pj_trips_byaccess

,REWEIGHTED_LINKED
ACCESS_MODE_PA,
KNR,2709.3
PNR,3352.4
Walk,16568.8


In [18]:
project_trips = df[df['TRIPS_ON_PROJECT']=='Yes']

### Table 3 in Report 

Table 3 Mid-Coast Project Trips, 2023 Transit On-Board Survey

In [19]:
pivot = pd.pivot_table(
    project_trips,
    index=['AUTO_OWNERSHIP', 'ACCESS_MODE_PA'],
    columns='TRIP_PURPOSE_AGG',
    values='REWEIGHTED_LINKED',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
)

# Reset index for better readability
pivot = pivot[['HBW', 'HBO', 'NHB', 'Total']]
pivot.reset_index().round(1).to_csv(output_dir + 'Table_3_Project_Trips_High_Level_Summary.csv',index=False)
pivot.round(1)

TRIP_PURPOSE_AGG                    HBW      HBO     NHB    Total
AUTO_OWNERSHIP   ACCESS_MODE_PA                                  
None (0)         KNR              107.2    391.1   368.5    866.8
                 PNR               72.6     12.3     0.0     84.9
                 Walk            1630.4   3533.2  1592.6   6756.2
One (1)          KNR              608.4    254.8     0.9    864.1
                 PNR              640.6    519.9     0.5   1160.9
                 Walk            1709.8   2122.7   885.8   4718.3
Two or more (2+) KNR              284.0    671.3    23.1    978.4
                 PNR              914.4   1192.1     0.0   2106.6
                 Walk            1615.2   2329.9  1149.2   5094.3
Total                            7582.5  11027.4  4020.6  22630.5

In [20]:
df.to_csv(interim_dir + 'Interim_2023_OnBoardSurvey_Dataset_1.csv',index=False)